# Day113-115: ORQA (Dist-S) inference on Colab, inside an isolated venv

Day113-114 installed ORQA's 2024-era pinned stack straight into Colab's shared system Python and fought Colab's own preinstalled packages until the runtime broke (see `README.md`). This version never installs anything ORQA-related into Colab's Python:

- **Colab kernel (Python 3.13)**: only orchestration and file I/O -- clone, download, frame extraction, printing results.
- **`/content/orqa-venv` (Python 3.10)**: every ORQA dependency, installed in one resolver pass. Inference runs as a separate script (`orqa_infer.py`) executed by the venv's own interpreter.

Why the venv needs a *different Python*, not just `python -m venv`: a venv created from Colab's Python inherits Python 3.13, but `torch==2.4.1` only ships wheels up to Python 3.12 (`open3d==0.18.0` and the prebuilt `torch-scatter` wheels likewise stop well before 3.13). The ORQA authors' own environment was Python 3.10 (their README mentions `ENV/lib/python3.10/site-packages`). `uv` downloads a standalone CPython 3.10 build and creates the venv from it.

Why the kernel can't just `import` from the venv: compiled extensions built for 3.10 (torch, numpy, spconv) can't be loaded into a 3.13 interpreter. So the model code lives in a script run by `/content/orqa-venv/bin/python`.

**Before running**: Runtime -> Disconnect and delete runtime (fresh VM), then Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all. Every step raises an error and stops if it fails, so the first red cell is the one to look at.

Expected time on a T4: ~10 min install, ~5 min downloads, a few minutes model load + inference.

In [ ]:
import os, sys, subprocess

def run(cmd):
    # Run a shell command, stream its output into the notebook, and STOP the notebook on failure.
    # (`!cmd` does not raise on a non-zero exit code, which is how Day113's silent failures slipped through.)
    print(f"$ {cmd}\n", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {cmd}")

os.environ["WANDB_DISABLED"] = "true"   # inherited by every subprocess below
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # progress bars flood the streamed log

print("Colab kernel Python:", sys.version)  # expected 3.13.x -- this is why we need a separate 3.10 venv
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
run("free -g | head -2; df -h /content | tail -1")

## 1. Clone ORQA (pinned commit)

Pinned to the exact commit whose code this notebook was checked against, so an upstream change can't silently alter the loading logic.

In [ ]:
ORQA_COMMIT = "12c97f985aa865f72263d7095ed062cf6a8bc7e0"  # egeozsoy/ORQA HEAD as of 2025-12-04
if not os.path.isdir("/content/ORQA/.git"):
    run("git init -q /content/ORQA")
    run("git -C /content/ORQA remote add origin https://github.com/egeozsoy/ORQA.git")
    run(f"git -C /content/ORQA fetch -q --depth 1 origin {ORQA_COMMIT}")
    run("git -C /content/ORQA checkout -q FETCH_HEAD")
run("git -C /content/ORQA log -1 --format='ORQA commit: %H (%cd)'")

## 2. Create the isolated Python 3.10 venv with `uv`

`uv` is a single self-contained binary with no Python dependencies, so putting it into Colab's Python can't conflict with anything there (and newer Colab images already ship it).

In [ ]:
VENV = "/content/orqa-venv"
VENV_PY = f"{VENV}/bin/python"
run("which uv || pip install -q uv")
run("uv --version")
if not os.path.exists(VENV_PY):
    run(f"uv venv --python 3.10 {VENV}")
run(f"{VENV_PY} --version")

## 3. Install ORQA's dependencies into the venv -- one resolver pass

Pins follow ORQA's `requirements.txt` + `Qwen2-VL/LLaMA-Factory/requirements.txt`, restricted to what image+text inference actually imports. Deliberate differences, each with a reason:

| Package | ORQA pin | Here | Why |
|---|---|---|---|
| numpy | 1.23.0 | 1.26.4 | LLaMA-Factory itself only requires `numpy<2.0.0`; 1.26.4 is the last 1.x and avoids "built against newer numpy" ABI errors from other wheels. Numerics come from torch, not numpy. |
| flash-attn | 2.6.1 | not installed | FlashAttention-2 needs Ampere (sm80+); the T4 is Turing (sm75). We use ORQA's own non-FA2 fallback (`eager`), see step 7. |
| spconv | `spconv-cu117` | `spconv-cu121` | Matches torch 2.4.1's CUDA 12.1 build. |
| torch-scatter | via conda | pyg prebuilt wheel `+pt24cu121` | Prebuilt, nothing to compile. |
| addict | (not listed) | latest | Imported by ORQA's `pointtransformerv3.py` but missing from `requirements.txt`. |
| tqdm, wandb, pytorch-lightning, uvloop, editdistance | pinned | not installed / left to resolver | Training/eval-loop only. (`tqdm==4.61.0` also conflicts with `datasets<=3.1.0`, which needs tqdm>=4.66.3.) |

**Why spconv / torch-scatter even for image-only inference**: ORQA's `modeling_qwen2_vl.py` imports `pointtransformerv3.py` at module level (`import spconv.pytorch`, `import torch_scatter`), and `ImageEmbeddingPooler.__init__` builds a `PointTransformerV3` unconditionally. The point-cloud branch is deleted right after loading, but import and construction still have to succeed.

Output is intentionally *not* silenced: Day113's combined install pinned `numpy==2.0.2` while LLaMA-Factory's `requirements.txt` says `numpy<2.0.0` -- an unsatisfiable request that likely made pip install nothing, which `-q` hid (and would explain the later `No module named 'trl'`).

In [ ]:
packages = [
    "torch==2.4.1", "torchvision==0.19.1",
    "transformers==4.46.1", "qwen-vl-utils==0.0.2", "bitsandbytes==0.44.1",
    "numpy==1.26.4", "open3d==0.18.0", "opencv-python==4.10.0.84",
    "timm==1.0.12", "torchinfo==1.8.0", "json-tricks==3.17.3", "addict",
    "spconv-cu121", "torch-scatter==2.1.2+pt24cu121",
]
pkg_args = " ".join(f'"{p}"' for p in packages)
run(f"uv pip install --python {VENV_PY} "
    f"--find-links https://data.pyg.org/whl/torch-2.4.0+cu121.html "
    f'{pkg_args} -e "/content/ORQA/Qwen2-VL/LLaMA-Factory[torch,metrics]"')

## 4. Verify the venv before touching the model

If anything fails here, fix it here -- not later through a model-loading traceback.

In [ ]:
%%writefile /content/check_env.py
import sys
import torch, torchvision, transformers, trl, peft, accelerate, numpy, open3d, spconv, torch_scatter
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU visible -- set Runtime -> Change runtime type -> T4 GPU"
print("GPU         ", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))
for m in (torchvision, transformers, trl, peft, accelerate, numpy, open3d, spconv):
    print(f"{m.__name__:12s}", m.__version__)
assert transformers.__version__ == "4.46.1", transformers.__version__
assert numpy.__version__.startswith("1."), numpy.__version__
# ORQA's own model module -- the import chain that needs spconv / torch_scatter / open3d
from llamafactory.model.qwen2_vl.modeling_qwen2_vl import Qwen2VLForConditionalGeneration, make_smaller
from llamafactory.data import get_template_and_fix_tokenizer, SFTDataCollatorWith4DAttentionMask
print("ORQA / LLaMA-Factory imports OK")

In [ ]:
run(f"uv pip check --python {VENV_PY}")
run(f"{VENV_PY} /content/check_env.py")

## 5. Download the Dist-S checkpoint

Dist-S is **not a LoRA adapter** (Day113's notebook treated it as one). Its `config.json` describes a full, shrunken Qwen2-VL: LLM hidden size 768 (vs 1536), 8 of 28 LLM layers kept (`depthreduce4`), plus ORQA's custom visual pooler. The zip holds:

- `model.safetensors` (~1.9 GB, bf16, ~938M params): the whole student -- 8 LLM layers, embeddings, and the full visual tower incl. `visual.image_pooler`
- `visual_block.pt` (~0.7 GB): visual weights the authors additionally load on top. **The released file is truncated** at exactly 700 MiB (734,003,200 bytes; Day115 run: `torch.load` fails with "failed finding central directory"). Because PKD freezes the whole visual tower (`only_llm=True`), it should duplicate the visual weights in `model.safetensors`; step 7's script checks this on every tensor still readable from the truncated file and stops if any differ.
- `reduction_percantage.txt` (`0.5`): hidden-size reduction the architecture must be rebuilt with (filename typo is the authors')
- tokenizer / processor files, saved with transformers 4.46.1 -- used instead of today's Hugging Face copies of Qwen2-VL, which could have drifted

In [ ]:
ZIP = "qwen2vl_lora_sft_qlora_1000000_unfreeze8_0.5mmdrop_336res_578imgtoks_pkd_050_depthreduce4"
CKPT = f"/content/saves/{ZIP}/checkpoint-124806"
os.makedirs("/content/saves", exist_ok=True)
if not os.path.exists(f"{CKPT}/model.safetensors"):
    run(f'wget -nv -O /content/saves/dist_s.zip "https://huggingface.co/egeozsoy/ORQA/resolve/main/checkpoints/{ZIP}.zip?download=true"')
    run("unzip -q -o /content/saves/dist_s.zip -d /content/saves && rm /content/saves/dist_s.zip")
run(f"ls -la {CKPT}")
run(f"cat {CKPT}/reduction_percantage.txt")

## 6. Test images

Two images, chosen to separate "the pipeline is broken" from "the model is out of its domain":

1. **Room-level OR photo (closer to ORQA's domain)** -- "Operating Room", National Cancer Institute, photographer John Crawford, **public domain**, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Operating_room.jpg). Two surgeons and a nurse, instrument table, OR lights, anaesthesia equipment. Still not ORQA's exact setting (4D-OR/MM-OR use fixed ceiling cameras; this is a hand-held photo), so treat it as "near-domain".
2. **Endoscope frame (out of domain)** -- frame 50 of the stomach-phantom video this repo used on Day106. ORQA never saw in-body footage.

Every image in `/content/test_images/` gets asked every question.

In [ ]:
import cv2, urllib.request
os.makedirs("/content/test_images", exist_ok=True)

# 1. room-level OR photo (public domain). Wikimedia rejects requests without a descriptive User-Agent.
req = urllib.request.Request(
    "https://commons.wikimedia.org/wiki/Special:FilePath/Operating_room.jpg?width=960",
    headers={"User-Agent": "surgical-video-ai-learning/1.0 (educational ORQA inference test)"},
)
with urllib.request.urlopen(req) as r, open("/content/test_images/or_room_nci.jpg", "wb") as f:
    f.write(r.read())
assert cv2.imread("/content/test_images/or_room_nci.jpg") is not None, "OR photo download is not a valid image"

# 2. stomach-phantom endoscope frame (Day106 data)
url = (
    "https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-Open-H-Embodiment/resolve/main/"
    "Endoscopy/cuhk/openh_dataset_full/find_greater_curvature/videos/chunk-000/"
    "observation.images.endo%E4%B8%89/episode_000000.mp4"
)
urllib.request.urlretrieve(url, "/content/test_episode.mp4")
cap = cv2.VideoCapture("/content/test_episode.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, 50)
ok, frame = cap.read()
cap.release()
assert ok, "could not read frame 50 of the endoscope video"
cv2.imwrite("/content/test_images/phantom_endoscope_f050.jpg", frame)

from IPython.display import Image as IPImage, display
for name in sorted(os.listdir("/content/test_images")):
    print(name, cv2.imread(f"/content/test_images/{name}").shape)
    display(IPImage(f"/content/test_images/{name}", width=400))

## 7. Inference script (runs inside the venv)

Mirrors the authors' own code path rather than LLaMA-Factory's generic `ChatModel`:

- **Processor patch** copies the `sys.modules` override at the top of `scene_graph_prediction/main.py`. Without it, transformers silently uses its *stock* `Qwen2VLImageProcessor`, which doesn't produce the `pixel_values_to_batch_idx` ORQA's fixed-578-token pooler needs, and preprocessing fails.
- **Loading** follows `load_pretrained_model` in `Qwen2-VL/LLaMA-Factory/src/llamafactory/model/qwen2_vl/qwen2_vl_helpers.py` (the `_pkd` branch).
- **Preprocessing / generation** follows `ORQAWrapperQA` (`scene_graph_prediction/scene_graph_helpers/model/scene_graph_prediction_model_oracle.py`) and `web_demo_orqa.py`.

Why not `ChatModel` (Day113's approach): LLaMA-Factory's inference loader loads `visual_block.pt` only when a *training* argument (`previous_model_weights`) is set (`model/loader.py:173`), and has no notion of the PKD shrink step. Even if it had loaded, ORQA's trained visual pooler would have silently stayed at random init.

Deviations from the authors' loader, all forced by this setup:

1. **Teacher stand-in**: the authors first load a local `..._pkd_teacher` checkpoint -- *not released on Hugging Face* -- only as a template, shrink it (`make_smaller`), then overwrite the student with `model.safetensors` via `load_state_dict(strict=False)`. We use `Qwen/Qwen2-VL-2B-Instruct` as the template. This is only harmless if the checkpoint overwrites every student parameter, so the script (a) asserts the rebuilt architecture matches the checkpoint's own `config.json`, and (b) counts every parameter the checkpoint did *not* overwrite. Non-empty (b) means the output isn't purely ORQA.
2. **Attention**: `eager` instead of `flash_attention_2` (T4 can't run FA2). `eager` is the authors' own fallback in the same function when CUDA is absent.
3. **dtype**: weights are bf16; the T4 has no native bf16, so the model runs in fp32 there (bf16 automatically on Ampere+).
4. **Tokenizer/processor**: loaded from the checkpoint folder itself (authors' wrapper loads them from the Qwen hub repo).
5. **flash_attn placeholder for the point-cloud branch**: ORQA's `PointTransformerV3` asserts `flash_attn` is importable in its constructor, even though it only calls it in `forward`. That branch is deleted immediately after construction (same as the authors' code), so a placeholder object gets past the constructor and is never called -- the script asserts the branch is gone.

In [ ]:
%%writefile /content/orqa_infer.py
"""ORQA Dist-S image+text inference. Run with /content/orqa-venv/bin/python (Python 3.10 venv)."""
import sys
import types

# ---- 0. Use ORQA's own processor classes (verbatim from scene_graph_prediction/main.py) ----
# Must run before anything asks transformers for Qwen2VLImageProcessor.
from llamafactory.model.qwen2_vl.image_processing_qwen2_vl import Qwen2VLImageProcessor
from llamafactory.model.qwen2_vl.processing_qwen2_vl import Qwen2VLProcessor

local_qwen2_vl_processing = types.ModuleType("transformers.models.qwen2_vl.processing_qwen2_vl")
local_qwen2_vl_processing.Qwen2VLProcessor = Qwen2VLProcessor
local_qwen2_vl_image_processing = types.ModuleType("transformers.models.qwen2_vl.image_processing_qwen2_vl")
local_qwen2_vl_image_processing.Qwen2VLImageProcessor = Qwen2VLImageProcessor
sys.modules["transformers.models.qwen2_vl.processing_qwen2_vl"] = local_qwen2_vl_processing
sys.modules["transformers.models.qwen2_vl.image_processing_qwen2_vl"] = local_qwen2_vl_image_processing

import gc
import glob
import io
import json
import os
import time
from copy import deepcopy
from types import SimpleNamespace

import torch
from transformers.modeling_utils import load_state_dict

from llamafactory.data import SFTDataCollatorWith4DAttentionMask, get_template_and_fix_tokenizer
from llamafactory.model import load_tokenizer
from llamafactory.model.qwen2_vl.modeling_qwen2_vl import Qwen2VLForConditionalGeneration, make_smaller
from llamafactory.model.qwen2_vl.qwen2_vl_helpers import _pkd_infer_distillation_information_from_model_path
import llamafactory.model.qwen2_vl.pointtransformerv3 as ptv3

# PointTransformerV3's attention asserts `flash_attn is not None` in its *constructor* (enable_flash=True by
# default), and ImageEmbeddingPooler always builds one. flash_attn is only called in its forward pass, and the
# whole point-cloud branch is deleted right after construction (step 1), so a placeholder is enough to get
# past construction; we assert below that the branch is really gone before any forward pass.
if ptv3.flash_attn is None:
    ptv3.flash_attn = types.SimpleNamespace(placeholder_never_called=True)

CKPT = sys.argv[1]
IMAGE_DIR = sys.argv[2]
OUT_JSON = sys.argv[3]

TEACHER_STANDIN = "Qwen/Qwen2-VL-2B-Instruct"  # architecture template only (see notebook step 7)
ATTN = "eager"                                 # authors' non-FA2 fallback in load_pretrained_model
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float32
FIX_NUMBER_OF_IMAGE_TOKENS = 578               # scene_graph_prediction/scene_graph_helpers/configs/orqa.json
IMAGE_RESOLUTION = 112896                      # 336*336, same config

# Questions phrased like ORQA's own training QA pairs (data/final_qa_pairs.zip), plus one open-ended
QUESTIONS = [
    "List all entities in the OR.",
    "What action is being performed at this time?",
    "Describe what you see in this image.",
]

def compare_truncated_visual_block(path, reference):
    """Read what survives of a truncated torch.save() file and compare each tensor with `reference`.

    torch.save writes an uncompressed zip: `<name>/data.pkl` (tensor metadata) first, then one entry per
    storage (`<name>/data/<key>`). A truncated file loses the zip's central directory at the end, but the
    entries before the cut are intact, so they can be walked via their local file headers.
    Returns (n_compared, n_different, n_not_comparable).
    """
    import pickle
    import struct

    raw = open(path, "rb").read()
    entries, pos = {}, 0
    while raw[pos:pos + 4] == b"PK\x03\x04":
        # torch writes each entry "streamed": sizes are 0 in the local header and come in a data
        # descriptor (PK\x07\x08, crc, sizes) right after the data. Find the descriptor whose recorded
        # size equals the distance from the data start -- that pins down the entry boundary.
        flags, method = struct.unpack("<HH", raw[pos + 6:pos + 10])
        name_len, extra_len = struct.unpack("<HH", raw[pos + 26:pos + 30])
        assert method == 0, "unexpected compressed entry"
        name = raw[pos + 30:pos + 30 + name_len].decode()
        start = pos + 30 + name_len + extra_len
        d, end = start, None
        while end is None:
            d = raw.find(b"PK\x07\x08", d)
            if d < 0 or d + 24 > len(raw):
                break  # the entry cut in half by the truncation
            if struct.unpack("<I", raw[d + 8:d + 12])[0] == d - start:
                end, desc_len = d, 16
            elif struct.unpack("<Q", raw[d + 8:d + 16])[0] == d - start:
                end, desc_len = d, 24  # zip64 descriptor
            else:
                d += 1
        if end is None:
            break
        entries[name] = raw[start:end]
        pos = end + desc_len

    dtypes = {"BFloat16Storage": torch.bfloat16, "FloatStorage": torch.float32,
              "HalfStorage": torch.float16, "LongStorage": torch.int64}

    class MetaOnly(pickle.Unpickler):  # rebuild tensor *descriptions*, not tensors
        def find_class(self, module, name):
            if module == "torch" and name in dtypes:
                return dtypes[name]
            if module == "torch._utils" and name == "_rebuild_tensor_v2":
                return lambda storage, offset, size, stride, *args: (storage, offset, tuple(size), tuple(stride))
            if module == "torch._utils" and name == "_rebuild_parameter":
                return lambda data, *args: data
            return super().find_class(module, name)

        def persistent_load(self, pid):  # ('storage', dtype, key, location, numel)
            return pid[1], pid[2]

    pkl_name = next(n for n in entries if n.endswith("/data.pkl"))
    prefix = pkl_name[:-len("data.pkl")]
    meta = MetaOnly(io.BytesIO(entries[pkl_name])).load()

    n_cmp = n_diff = n_missing = 0
    for key, ((dtype, storage_key), offset, size, stride) in meta.items():
        data = entries.get(f"{prefix}data/{storage_key}")
        if data is None or key not in reference:
            n_missing += 1
            continue
        t = torch.frombuffer(bytearray(data), dtype=dtype).as_strided(size, stride, offset)
        ref = reference[key]
        n_cmp += 1
        if t.shape != ref.shape or not torch.equal(t.to(ref.dtype), ref):
            n_diff += 1
            print("   differs:", key)
    return n_cmp, n_diff, n_missing


# ---- 1. Build the student architecture (qwen2_vl_helpers.load_pretrained_model, _pkd branch) ----
t0 = time.time()
model = Qwen2VLForConditionalGeneration.from_pretrained(TEACHER_STANDIN, torch_dtype=torch.bfloat16,
                                                        attn_implementation=ATTN)
del model.visual.image_pooler.temporal_cross_attention
del model.visual.image_pooler.temporal_segment_embedding
del model.visual.image_pooler.point_transformer
del model.visual.image_pooler.point_pooling
del model.visual.image_pooler.project_audio
fixed_depth_reduction, reduction_percentage = _pkd_infer_distillation_information_from_model_path(CKPT)
model.original_config = deepcopy(model.config)
student = make_smaller(model, reduction_percentage=reduction_percentage, only_llm=True,
                       fixed_depth_reduction=fixed_depth_reduction)
del model
gc.collect()
assert not hasattr(student.visual.image_pooler, "point_transformer"), "point-cloud branch still present"

# (a) the rebuilt architecture must match the checkpoint's own config.json
with open(os.path.join(CKPT, "config.json")) as f:
    ckpt_cfg = json.load(f)
rebuilt = {"hidden_size": student.config.hidden_size, "intermediate_size": student.config.intermediate_size,
           "mrope_section": list(student.config.rope_scaling["mrope_section"])}
expected = {"hidden_size": ckpt_cfg["hidden_size"], "intermediate_size": ckpt_cfg["intermediate_size"],
            "mrope_section": ckpt_cfg["rope_scaling"]["mrope_section"]}
print("rebuilt architecture:", rebuilt, "| checkpoint config:", expected)
assert rebuilt == expected, "rebuilt student architecture does not match the checkpoint config"

# ---- 2. Overwrite with the released weights, and (b) check what was NOT overwritten ----
state_dict = load_state_dict(os.path.join(CKPT, "model.safetensors"))
res = student.load_state_dict(state_dict, strict=False)
loaded = set(state_dict) - set(res.unexpected_keys)
del state_dict
print(f"model.safetensors: unexpected keys={len(res.unexpected_keys)} {res.unexpected_keys[:5]}")

visual_block_path = os.path.join(CKPT, "visual_block.pt")
visual_block_status = "absent"
if os.path.exists(visual_block_path):
    try:
        visual_block = torch.load(visual_block_path, map_location="cpu")
    except RuntimeError as e:
        # The released Dist-S visual_block.pt is truncated at exactly 700 MiB (734,003,200 bytes), so torch.load
        # can't find the zip's central directory. PKD trains with only_llm=True, which freezes every visual.*
        # parameter (modeling_qwen2_vl.py make_smaller), so this file should duplicate the visual weights already
        # in model.safetensors. Verify that on every tensor still readable from the truncated file.
        print(f"visual_block.pt unreadable by torch.load ({e}); comparing its readable part with model.safetensors")
        visual_block = None
        n_cmp, n_diff, n_missing = compare_truncated_visual_block(visual_block_path, student.visual.state_dict())
        print(f"visual_block.pt (truncated): compared={n_cmp}, differ={n_diff}, not comparable (cut off, or key not in model)={n_missing}")
        assert n_cmp > 0 and n_diff == 0, "truncated visual_block.pt disagrees with model.safetensors -- cannot skip it"
        visual_block_status = f"truncated; {n_cmp} readable tensors identical to model.safetensors, {n_missing} not comparable"
    if visual_block is not None:
        res_v = student.visual.load_state_dict(visual_block, strict=False)
        loaded |= {"visual." + k for k in visual_block if k not in res_v.unexpected_keys}
        print(f"visual_block.pt: {len(visual_block)} tensors, unexpected keys={len(res_v.unexpected_keys)}")
        visual_block_status = "loaded"
        del visual_block
gc.collect()

lm_head_tied = student.lm_head.weight.data_ptr() == student.model.embed_tokens.weight.data_ptr()
not_loaded = [k for k in student.state_dict() if k not in loaded and not (k == "lm_head.weight" and lm_head_tied)]
print(f"lm_head tied to embed_tokens: {lm_head_tied}")
print(f"student params NOT covered by the checkpoint: {len(not_loaded)}")
for k in not_loaded[:30]:
    print("   ", k)
n_params = sum(p.numel() for p in student.parameters())
print(f"student: {n_params / 1e6:.1f}M params, {len(student.model.layers)} LLM layers, "
      f"hidden={student.config.hidden_size}, LLM attn={student.config._attn_implementation}, "
      f"vision attn={student.visual.config._attn_implementation}")

student = student.to(device=DEVICE, dtype=DTYPE).eval()
print(f"loaded in {time.time() - t0:.0f}s, dtype={DTYPE}, "
      f"GPU mem allocated={torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ---- 3. Tokenizer / template / collator (ORQAWrapperQA.__init__) ----
student.visual.image_pooler.fix_number_of_image_tokens = FIX_NUMBER_OF_IMAGE_TOKENS
student.visual.image_pooler.use_past_visual_embeds = False
model_args = SimpleNamespace(model_name_or_path=CKPT, cache_dir=None, model_revision=None, hf_hub_token=None,
                             use_fast_tokenizer=True, split_special_tokens=False, new_special_tokens=None,
                             image_resolution=IMAGE_RESOLUTION, video_resolution=128 * 128, video_fps=2.0, video_maxlen=64)
data_args = SimpleNamespace(template="qwen2_vl", train_on_prompt=False, ignore_pad_token_for_loss=True, tool_format=None,
                            fix_number_of_image_tokens=FIX_NUMBER_OF_IMAGE_TOKENS, use_past_visual_embeds=False)
tokenizer_module = load_tokenizer(model_args)
tokenizer_module["tokenizer"].padding_side = "left"
tokenizer, processor = tokenizer_module["tokenizer"], tokenizer_module["processor"]
# load_tokenizer swallows processor errors (logs at debug level, returns None) -- fail loudly instead
assert processor is not None, "processor failed to load"
print("image processor class:", type(processor.image_processor).__module__, type(processor.image_processor).__name__)
assert type(processor.image_processor).__module__.startswith("llamafactory"), "ORQA's image processor patch did not take effect"
template = get_template_and_fix_tokenizer(tokenizer, data_args)
collator = SFTDataCollatorWith4DAttentionMask(template=template, label_pad_token_id=-100, block_diag_attn=False,
                                              attn_implementation=student.config._attn_implementation,
                                              compute_dtype=DTYPE, **tokenizer_module)


def ask(image_path, question):
    # web_demo_orqa.py: "<image>" prepended to the question, empty assistant turn as generation prompt
    messages = [{"role": "user", "content": "<image>" + question}, {"role": "assistant", "content": ""}]
    images = [image_path]
    processed = template.mm_plugin.process_messages(messages, images, [], processor)
    input_ids, _ = template.encode_oneturn(tokenizer, processed, system=None, tools=None)
    batch = collator([{"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "images": images}])
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            v = v.to(DEVICE)
            batch[k] = v.to(DTYPE) if v.is_floating_point() else v
    for k, v in (getattr(batch, "_multimodal_extras", None) or {}).items():  # pcs/audios/past_visual_embeds (all empty here)
        batch.data[k] = v
    with torch.inference_mode():
        start = time.time()
        out = student.generate(**batch, max_new_tokens=300, do_sample=False, use_cache=True)  # ORQAWrapperQA settings
        dt = time.time() - start
    new_tokens = out[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip(), len(new_tokens), dt


results = []
for image_path in sorted(glob.glob(os.path.join(IMAGE_DIR, "*"))):
    for q in QUESTIONS:
        answer, n_tok, dt = ask(image_path, q)
        results.append({"image": os.path.basename(image_path), "question": q, "answer": answer,
                        "new_tokens": n_tok, "seconds": round(dt, 2)})
        print(f"\n[{os.path.basename(image_path)}] Q: {q}\nA: {answer}\n({n_tok} tokens, {dt:.1f}s)", flush=True)

meta = {"checkpoint": CKPT, "teacher_standin": TEACHER_STANDIN, "dtype": str(DTYPE),
        "visual_block": visual_block_status,
        "llm_attn": student.config._attn_implementation, "vision_attn": student.visual.config._attn_implementation,
        "params_not_covered_by_checkpoint": not_loaded, "gpu": torch.cuda.get_device_name(0)}
with open(OUT_JSON, "w") as f:
    json.dump({"meta": meta, "results": results}, f, indent=2, ensure_ascii=False)
print("\nsaved", OUT_JSON)

## 8. Run it

Read the two check lines first -- "rebuilt architecture" must match, and "NOT covered by the checkpoint" should be 0. Anything listed there came from the Qwen2-VL-2B-Instruct stand-in (or random init), not from ORQA.

In [ ]:
run(f"cd /content && {VENV_PY} -u /content/orqa_infer.py {CKPT} /content/test_images /content/orqa_outputs.json")

In [ ]:
import json
out = json.load(open("/content/orqa_outputs.json"))
print(json.dumps(out["meta"], indent=2))
for r in out["results"]:
    print(f"\n[{r['image']}] {r['question']}\n  -> {r['answer']}")

# Keep the raw outputs: download and commit next to this notebook
from google.colab import files
files.download("/content/orqa_outputs.json")

## Notes for interpreting the output

- **Check the two verification lines before reading any answer.** A non-empty "NOT covered" list means part of the network is Qwen2-VL-2B-Instruct (or random), and the answers say nothing clean about ORQA.
- **Compare the two images against each other**, not against an absolute bar:
  - Sensible, OR-vocabulary answers on the room photo + generic/hallucinated OR answers on the endoscope frame -> pipeline works; ORQA is specialized to room-level views (consistent with the paper).
  - Incoherent or repetitive text on *both* -> pipeline problem (attention backend, dtype, preprocessing), not a domain gap.
- ORQA's training answers are short templated phrases ("We can list the entities as ..."), so terse answers are expected; the open-ended "Describe" question is outside its training format.
- Runtime -> Disconnect and delete runtime when done.